In [88]:
import sys
import os
import numpy as np
import MDAnalysis as mda
from MDAnalysis.analysis import align
from MDAnalysis.analysis import distances

import matplotlib.pyplot as plt


In [123]:
#Chatgpt prompt:
"""write a python function that takes as its arguments a path to a pdb file, an mdanalysis atomgroup, and a list of numbers equal in length to the atom group, 
and saves a copy of the pdb file with the b factors of the atoms in the atom group set equal to the numbers in the list. 
Match atoms using residue names and numbers since the indices do not match between the atomgroup and pdb file"""


def write_bfactors_by_residue_match(pdb_path, atomgroup, values, output_pdb=None):
    """
    Write a copy of a PDB file with B-factors set for atoms matching an AtomGroup.
    Matching is done using (resname, resid, atom name), NOT indices.

    Parameters
    ----------
    pdb_path : str
        Path to input PDB file.
    atomgroup : MDAnalysis AtomGroup
        AtomGroup derived from (possibly different) structure.
    values : array-like
        Numbers equal in length to atomgroup.
    output_pdb : str
        Path to output PDB file.
    """

    if output_pdb is None:
        base = os.path.splitext(pdb_path)[0]
        output_pdb = base + "_bfactored.pdb"

    values = np.asarray(values)

    if len(atomgroup) != len(values):
        raise ValueError("Length of values must equal AtomGroup length.")

    # Load fresh universe from PDB to preserve original ordering
    u = mda.Universe(pdb_path)

    # Ensure tempfactors exist
    if not hasattr(u.atoms, "tempfactors"):
        u.add_TopologyAttr("tempfactors")

    # Start from existing B-factors (or zeros)
    try:
        b = u.atoms.tempfactors.copy()
    except AttributeError:
        b = np.zeros(len(u.atoms))

    # Build lookup dictionary from PDB universe
    pdb_lookup = {}
    for atom in u.atoms:
        key = (atom.resname, atom.resid, atom.name)
        pdb_lookup[key] = atom.index

    # Assign B-factors by matching keys
    for atom, value in zip(atomgroup, values):
        key = (atom.resname, atom.resid, atom.name)

        if key not in pdb_lookup:
            raise ValueError(
                f"Atom not found in PDB: resname={atom.resname}, "
                f"resid={atom.resid}, name={atom.name}"
            )

        pdb_index = pdb_lookup[key]
        b[pdb_index] = value

    # Reassign full array (required by MDAnalysis)
    u.atoms.tempfactors = b

    # Write output
    with mda.Writer(output_pdb, multiframe=False) as W:
        W.write(u.atoms)


In [51]:
def tmd_query():
    segment_resis = [[77, 149], [192, 245], [298, 362], [988, 1034], [857, 889], [900, 942], [1094, 1154]]
    print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"

def binding_site_query():
    segment_resis = [[221, 245], [298, 328], [857, 886], [911, 942]]
    print("color blue, " + " or ".join([f"resi {sr[0]}-{sr[1]}" for sr in segment_resis]))

    segment_resis_all = [i for sr in segment_resis for i in range(sr[0], sr[1]+1)]
    query = " or ".join([f"resid {sr}" for sr in segment_resis_all])

    #indices = frame.top.select(f"protein and ({query})")

    # print_indices = frame.top.select(f"protein and name CA and ({query})")
    # print("+".join([str(i+1) for i in print_indices]))
    return f"protein and ({query})"

#binding_site_query()

In [161]:
def main():
    # ============================
    # LOAD TRAJECTORY
    # ============================

    ref_frame_path_x01 = '/media/X01Raid01/Data_Backup/home/csheen/cftr-project/wstp_cftr_1_degrabo/bstates/input/min.gro'
    ref_frame_path = '/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.gro'

    ref_frame = mda.Universe(ref_frame_path)

    gro_file = "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.gro"
    xtc_file = "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/001913-000187-trj-pbcmol-centered-tmd-rot-s10.xtc"

    # Frame index to analyze
    frame_index = -1

    u = mda.Universe(gro_file, xtc_file)

    u.trajectory[frame_index]

    align.AlignTraj(u, ref_frame, select=f"{tmd_query()} and name CA")

    return u
    
u = main()

print(u.trajectory.n_frames)

color blue, resi 77-149 or resi 192-245 or resi 298-362 or resi 988-1034 or resi 857-889 or resi 900-942 or resi 1094-1154
573


In [162]:
def contacts_bin(group_1, group_2, cutoff=5.0):
    dists = distances.distance_array(group_1, group_2, box=u.dimensions)
    print(dists.shape)
    contacts = np.where(dists < cutoff, 1, 0)
    contact_bin = np.where(np.sum(contacts, axis=1) > 0, 1, 0)

    return contact_bin

In [163]:
# #!/usr/bin/env python3

def select_heavy_atoms(u):
    """Select heavy atoms based on a given selection string."""

    print(u.trajectory.n_frames)

    # ============================
    # SELECTION PARAMETERS
    # ============================

    # List of protein residue numbers (edit as needed)
    #protein_resids = [10, 25, 48]   # <-- USER FILLS THIS
    protein_atoms = f"{tmd_query()} and not name H*"
    #protein_atoms = "resid 207 and resname GLN and not name H*"

    # Ligand residue name
    ligand_resname = "LJP"          # <-- USER FILLS THIS

    #not actually phosphates anymore
    # Lipid phosphate atom name (common choices: "P", "PO4", etc.)
    phosphate_selection = "(resname PA or resname OL) and name C12"  #"resname PC and name P31"

    #membrane_core_buffer = 0


    # ============================
    # PROTEIN HEAVY ATOMS
    # ============================

    #resid_string = " ".join(str(r) for r in protein_resids)

    protein_sel = u.select_atoms(protein_atoms)

    # ============================
    # LIGAND HEAVY ATOMS
    # ============================

    ligand_sel = u.select_atoms(
        f"resname {ligand_resname} and not name H*"
    )

    # ============================
    # MEMBRANE PHOSPHATES
    # ============================

    phosphates = u.select_atoms(phosphate_selection)

    if len(phosphates) == 0:
        raise ValueError("No phosphate atoms found. Check lipid resnames and phosphate atom name.")

    print(phosphates.positions.shape)

    z_positions = phosphates.positions[:, 2]
    z_mean = np.mean(z_positions)

    # Separate into upper and lower leaflets
    upper_leaflet = phosphates[z_positions > z_mean]
    lower_leaflet = phosphates[z_positions <= z_mean]

    if len(upper_leaflet) == 0 or len(lower_leaflet) == 0:
        raise ValueError("Leaflet separation failed.")

    z_upper_avg = np.mean(upper_leaflet.positions[:, 2])
    z_lower_avg = np.mean(lower_leaflet.positions[:, 2])

    z_min = min(z_upper_avg, z_lower_avg)
    z_max = max(z_upper_avg, z_lower_avg)

    # ============================
    # WATERS INSIDE MEMBRANE
    # ============================

    waters = u.select_atoms("resname TP3 and name O")

    water_z = waters.positions[:, 2]

    inside_mask = (water_z > z_min) & (water_z < z_max)
    waters_inside = waters[inside_mask]

    #print(np.where(inside_mask))
    #water_in_inds = np.where(inside_mask)[0]
    print("color blue, index " + "+".join([str(i+1) for i in waters_inside.indices]))
    # ============================
    # OUTPUT SUMMARY
    # ============================

    print(f"Protein heavy atoms: {len(protein_sel)}")
    print(f"Ligand heavy atoms: {len(ligand_sel)}")
    print(f"Waters inside membrane (heavy atoms): {len(waters_inside)}")

    # water_protein_dists = distances.distance_array(protein_sel,
    #                          waters_inside,
    #                          box=u.dimensions)

    # water_protein_contacts = np.where(water_protein_dists < 5, 1, 0)
    # protein_water_contact_bin = np.where(np.sum(water_protein_contacts, axis=1) > 0, 1, 0)

    #water contacts are averaged across waters so we get a 1d contact array
    protein_water_contacts = contacts_bin(protein_sel, waters_inside, cutoff=5.0)
    ligand_water_contacts = contacts_bin(ligand_sel, waters_inside, cutoff=5.0)

    #protein-ligand contacts are not averaged so we get a 2d contact array
    protein_ligand_dists = distances.distance_array(protein_sel,
                             ligand_sel,
                             box=u.dimensions)

    protein_ligand_contacts = np.where(protein_ligand_dists < 5, 1, 0)

    # water_protein_contacts = contacts.Contacts(u, select=(protein_sel, waters_inside), refgroup=(protein_sel, waters_inside), radius=4.5)
    # water_protein_contacts.run()
    # print(water_protein_contacts.results)
    # Example: save indices if desired
    # np.save("protein_indices.npy", protein_sel.indices)
    # np.save("ligand_indices.npy", ligand_sel.indices)
    # np.save("waters_inside_indices.npy", waters_inside.indices)

    return waters_inside.positions, protein_water_contacts, ligand_water_contacts, protein_ligand_contacts, protein_sel, ligand_sel

wat_pos, pwc, lwc, plc, prot_sel, lig_sel = select_heavy_atoms(u)

573
color blue, resi 77-149 or resi 192-245 or resi 298-362 or resi 988-1034 or resi 857-889 or resi 900-942 or resi 1094-1154
(578, 3)
color blue, index 58357+58435+58771+59236+59359+60166+60289+61129+61195+61390+61711+62041+62935+63169+63319+63472+63649+64033+64510+64585+64999+65920+65959+66733+67015+67357+67975+68077+68113+68341+68590+68626+68764+69268+69376+70507+70534+71329+71425+71578+72037+72673+72712+73003+73339+73366+73405+73660+73963+74779+75094+75397+75796+76273+76459+76528+76738+76801+78145+78880+78958+79366+79444+79549+79915+80281+80488+81280+81394+82114+82165+82339+82576+82891+83149+83200+84133+84826+85369+85636+85684+86260+86344+86569+87106+87313+87877+87892+87895+88216+88627+88726+89107+89245+89350+89545+90013+90349+90487+91111+91291+91396+91600+91639+92173+92293+92344+92812+92986+93064+93163+94300+94810+95014+95107+95671+96034+96559+97060+97108+97312+97696+98026+98290+98320+99418+99517+99613+99823+100666+100693+100789+100831+101470+101527+102199+102712+102913+103111+10

In [164]:
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", prot_sel, pwc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_pwc.pdb")
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", lig_sel, lwc,
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_lwc.pdb")


In [145]:
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", lig_sel, np.sum(plc, axis=0),
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_plc_lig.pdb")
write_bfactors_by_residue_match("/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input.pdb", prot_sel, np.sum(plc, axis=1),
                                "/home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/topology/input_plc_prot.pdb")

/home/jonathan/anaconda3/envs/serpents/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:346: UserWarning: Unknown element P3 found for some atoms. These have been given an empty element record. If needed they can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn(wmsg)
/home/jonathan/anaconda3/envs/serpents/lib/python3.11/site-packages/MDAnalysis/topology/PDBParser.py:376: UserWarning: Unknown entry 1 encountered in formal charge field. This likely indicates that the PDB file is not fully standard compliant. The formalcharges attribute will not be populated.
  warnings.warn(wmsg)
/home/jonathan/anaconda3/envs/serpents/lib/python3.11/site-packages/MDAnalysis/coordinates/PDB.py:1154: UserWarning: Found no information for attr: 'formalcharges' Using default value of '0'
  warnings.warn("Found no information for attr: '{}'"
/home/jonathan/anaconda3/envs/serpents/lib/python3.11/site-packages/MDAnalysis/coordinates/PDB.py:120

In [ ]:
u.trajectory

<XTCReader /home/jonathan/Documents/grabelab/cftr/independent-partial-dissociation/nonlip_glpg_1/001913-000187-trj-pbcmol-centered-tmd-rot-s10.xtc with 573 frames of 179115 atoms>

In [8]:
tmd_query()

color blue, resi 77-149 or resi 192-245 or resi 298-362 or resi 988-1034 or resi 857-889 or resi 900-942 or resi 1094-1154


'protein and (resSeq 77 or resSeq 78 or resSeq 79 or resSeq 80 or resSeq 81 or resSeq 82 or resSeq 83 or resSeq 84 or resSeq 85 or resSeq 86 or resSeq 87 or resSeq 88 or resSeq 89 or resSeq 90 or resSeq 91 or resSeq 92 or resSeq 93 or resSeq 94 or resSeq 95 or resSeq 96 or resSeq 97 or resSeq 98 or resSeq 99 or resSeq 100 or resSeq 101 or resSeq 102 or resSeq 103 or resSeq 104 or resSeq 105 or resSeq 106 or resSeq 107 or resSeq 108 or resSeq 109 or resSeq 110 or resSeq 111 or resSeq 112 or resSeq 113 or resSeq 114 or resSeq 115 or resSeq 116 or resSeq 117 or resSeq 118 or resSeq 119 or resSeq 120 or resSeq 121 or resSeq 122 or resSeq 123 or resSeq 124 or resSeq 125 or resSeq 126 or resSeq 127 or resSeq 128 or resSeq 129 or resSeq 130 or resSeq 131 or resSeq 132 or resSeq 133 or resSeq 134 or resSeq 135 or resSeq 136 or resSeq 137 or resSeq 138 or resSeq 139 or resSeq 140 or resSeq 141 or resSeq 142 or resSeq 143 or resSeq 144 or resSeq 145 or resSeq 146 or resSeq 147 or resSeq 148 or r